# 11 — Homography: DLT estimation, normalization

**Needed for the assignment:** no (course / exam topic). **Study:** 2 h, **Practice:** 1.5 h.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Idea

A homography $H$ ($3\times3$, defined up to scale, 8 DoF) maps one plane to another: $\mathbf{u}'\sim H\mathbf{u}$. It appears when two images see the same *plane*, or when two images are taken from the same centre (panoramas). Each point correspondence gives 2 linear equations in the 9 entries of $H$. So 4 points are enough, and more points are solved with "$A\mathbf{h}=\mathbf{0}$, smallest eigenvector" (DLT).

## Key formulas

## Homography Estimation (DLT)

A homography maps points between two images using a \(3 \times 3\) matrix \(H\).

### 1. Point mapping

For a point \(\mathbf{u}=(u,v,1)^T\), the corresponding point is:

$$
\mathbf{u}' \sim H\mathbf{u}
$$

After dividing by the third coordinate:

$$
u'=\frac{h_{11}u+h_{12}v+h_{13}}{h_{31}u+h_{32}v+h_{33}}
$$

$$
v'=\frac{h_{21}u+h_{22}v+h_{23}}{h_{31}u+h_{32}v+h_{33}}
$$

### 2. DLT linear system

Rearranging the equations gives two rows for each corresponding point pair:

$$
A_i=
\begin{bmatrix}
u&v&1&0&0&0&-uu'&-vu'&-u'\\
0&0&0&u&v&1&-uv'&-vv'&-v'
\end{bmatrix}
$$

Stack the rows from all point correspondences to construct \(A\).

The homography parameters are:

$$
\mathbf{h}=
(h_{11},h_{12},h_{13},
h_{21},h_{22},h_{23},
h_{31},h_{32},h_{33})^T
$$

Solve:

$$
A\mathbf{h}=0
$$

Since the solution is defined up to scale, impose:

$$
\|\mathbf{h}\|=1
$$

The solution is the right singular vector of \(A\) corresponding to its smallest singular value.

### 3. Point normalization

Normalize the points before applying DLT to improve numerical stability.

For each image, calculate the centroid and the mean distance of the points from that centroid.

$$
s=\frac{\sqrt{2}}{\text{mean distance to centroid}}
$$

Construct the normalization transformation:

$$
T=
\begin{bmatrix}
s&0&-s\bar{u}\\
0&s&-s\bar{v}\\
0&0&1
\end{bmatrix}
$$

Apply separate transformations to the two images:

$$
\mathbf{u}_n=T_1\mathbf{u}
$$

$$
\mathbf{u}'_n=T_2\mathbf{u}'
$$

Estimate the normalized homography \(H_n\) using DLT.

### 4. Denormalization

Convert the normalized homography back to the original image coordinates:

$$
H=T_2^{-1}H_nT_1
$$

### 5. Error and refinement

The **algebraic error** measures how well the linear system is satisfied, but it has no direct geometric meaning.

For better accuracy, minimize the **transfer error**:

$$
\sum_i d(\mathbf{u}'_i,H\mathbf{u}_i)^2
$$

Here, \(d\) is the Euclidean distance between the observed point and the projected point (after converting homogeneous coordinates back to image coordinates).

This nonlinear optimization can be performed using **Levenberg–Marquardt**.

### 6. Outliers and RANSAC

When correspondences contain outliers, use **RANSAC** around DLT:

1. Randomly select 4 point correspondences.
2. Estimate a homography from those 4 pairs.
3. Count the inliers whose transfer error is below a threshold.
4. Repeat and keep the model with the best inlier support.
5. Re-estimate the homography using all inliers.

# Worked example (on paper)

The unit square $(0,0),(1,0),(0,1),(1,1)$ maps to $(1,1),(3,1),(1,3),(3,3)$ (scale 2, shift 1). Guess $H=\begin{bmatrix}2&0&1\\0&2&1\\0&0&1\end{bmatrix}$, i.e. $\mathbf{h}=(2,0,1,0,2,1,0,0,1)$.

**Rows for $(1,0)\to(3,1)$:** $[1,0,1,0,0,0,-3,0,-3]$ and $[0,0,0,1,0,1,-1,0,-1]$.\
Check the first row: $2\cdot1+1\cdot1-3\cdot1=0$ ✓. Second row: $0+1\cdot1-1\cdot1=0$ ✓. With 4 correspondences $A$ is $8\times9$ with rank 8, so this $\mathbf{h}$ is its exact null vector.

**Normalization matrix of the source points.** The centroid is $(0.5,0.5)$. Every point is at distance $\sqrt{0.5}\approx0.707$, so $s=\sqrt2/0.707=2$: $$T_1=\begin{bmatrix}2&0&-1\\0&2&-1\\0&0&1\end{bmatrix},\qquad\text{normalized points }(\pm1,\pm1).$$

# The same in Python / NumPy

In [2]:
import numpy as np

def normalization_matrix(pts):
    """lecture version: centroid to origin, average distance sqrt(2)"""
    mean = pts.mean(axis=0)
    s = np.sqrt(2) / np.mean(np.linalg.norm(pts - mean, axis=1))
    return np.array([[s, 0, -s * mean[0]], [0, s, -s * mean[1]], [0, 0, 1]])

def apply(H, pts):
    h = np.column_stack([pts, np.ones(len(pts))]) @ H.T
    return h[:, :2] / h[:, 2:3]

def dlt(p1, p2):
    """rows from lec01: [u v 1 0 0 0 -uu' -vu' -u'] and [0 0 0 u v 1 -uv' -vv' -v']"""
    A = []
    for (u, v), (u2, v2) in zip(p1, p2):
        A.append([u, v, 1, 0, 0, 0, -u * u2, -v * u2, -u2])
        A.append([0, 0, 0, u, v, 1, -u * v2, -v * v2, -v2])
    A = np.array(A)
    val, vec = np.linalg.eig(A.T @ A)
    val, vec = val.real, vec.real
    return vec[:, np.argmin(val)].reshape(3, 3)

# --- paper example: H maps the unit square (scale 2, shift 1) ---
src = np.array([[0, 0], [1, 0], [0, 1], [1, 1]], dtype=float)
dst = np.array([[1, 1], [3, 1], [1, 3], [3, 3]], dtype=float)
H = dlt(src, dst)
print("H (DLT, scaled so H[2,2]=1) =\n", np.round(H / H[2, 2], 6))
print("T for the source points =\n", normalization_matrix(src))
print("normalized source points =\n", apply(normalization_matrix(src), src))

# --- noisy projective example in pixel units: with vs without normalization ---
rng = np.random.default_rng(0)
H_true = np.array([[0.9, 0.05, 30], [-0.04, 1.1, 10], [1e-4, 2e-4, 1]])
p1 = rng.uniform(0, 1000, size=(30, 2))
p2 = apply(H_true, p1) + rng.normal(0, 0.5, size=(30, 2))

H_raw = dlt(p1, p2)
T1, T2 = normalization_matrix(p1), normalization_matrix(p2)
H_n = dlt(apply(T1, p1), apply(T2, p2))
H_den = np.linalg.inv(T2) @ H_n @ T1                  # denormalize

def cond_A(p, q):
    A = []
    for (u, v), (u2, v2) in zip(p, q):
        A.append([u, v, 1, 0, 0, 0, -u * u2, -v * u2, -u2])
        A.append([0, 0, 0, u, v, 1, -u * v2, -v * v2, -v2])
    A = np.array(A)
    return np.linalg.cond(A.T @ A)
print(f"cond(A^T A) raw = {cond_A(p1, p2):.1e},  normalized = {cond_A(apply(T1, p1), apply(T2, p2)):.1e}")
for name, Hx in [("raw       ", H_raw), ("normalized", H_den)]:
    err = np.linalg.norm(apply(Hx, p1) - p2, axis=1).mean()
    print(f"{name}: mean transfer error = {err:.3f} px")
print("H estimated (normalized) =\n", np.round(H_den / H_den[2, 2], 5))

H (DLT, scaled so H[2,2]=1) =
 [[2. 0. 1.]
 [0. 2. 1.]
 [0. 0. 1.]]
T for the source points =
 [[ 2.  0. -1.]
 [ 0.  2. -1.]
 [ 0.  0.  1.]]
normalized source points =
 [[-1. -1.]
 [ 1. -1.]
 [-1.  1.]
 [ 1.  1.]]
cond(A^T A) raw = 6.4e+14,  normalized = 1.6e+06
raw       : mean transfer error = 0.654 px
normalized: mean transfer error = 0.652 px
H estimated (normalized) =
 [[ 0.8997  0.0497 30.2209]
 [-0.0399  1.0997 10.0345]
 [ 0.0001  0.0002  1.    ]]


Normalization drops the condition number from about $10^{14}$ to about $10^{6}$. In 64-bit floats both versions still give a good $H$ here, but with fewer points, more noise or `float32` the raw version breaks (lec01 calls normalization "obligatory").

## Course / lab link

Lab 6 has the same rows with opposite signs (which doesn't matter for $A\mathbf{h}=\mathbf{0}$). Its $s$ comes from the standard deviations instead of the mean distance; both are fine. Watch out: Lab 6 computes the normalized points but then passes the *raw* points to the DLT, and the denormalization line is commented out.

## Practice

Write your solution in the empty cell under each exercise. The **expected result** tells you what you should get, not how to get it. Grey *provided data* cells are already filled in; run them first.

### Exercise 1

Write the two rows of $A$ for $(2,1)\to(5,3)$, and check them with the $H$ of the example.

> **Expected result:** $[2,1,1,0,0,0,-10,-5,-5]$ and $[0,0,0,2,1,1,-6,-3,-3]$. Both dot products with $\mathbf h=(2,0,1,0,2,1,0,0,1)$ are $0$.

In [3]:
# Exercise 1: your solution


### Exercise 2

Compute $T$ by hand for the points $(0,0),(4,0),(4,2),(0,2)$.

> **Expected result:** Centroid $(2,1)$, every distance $\sqrt5$, $s=\sqrt2/\sqrt5\approx0.6325$; $T\approx\begin{bmatrix}0.6325&0&-1.2649\\0&0.6325&-0.6325\\0&0&1\end{bmatrix}$.

In [4]:
# Exercise 2: your solution


### Exercise 3

Estimate $H$ from the provided correspondences with normalization **and** denormalization. Report $H/H_{33}$ and the mean transfer error. (Optional: do the same with Lab 6's `features.txt` and compare with Lab 6's result.)

> **Expected result:** $H\approx\begin{bmatrix}1.1989&0.0999&-49.84\\0.0496&0.8995&20.11\\0.0002&-0.0001&1\end{bmatrix}$, mean transfer error $\approx0.509$ px.

In [5]:
# --- provided data (run me first) ---
# src : (30, 2) pixel coordinates in image 1, uniform in [0, 1000]^2
# dst : (30, 2) the same points in image 2 = H_true(src) + noise (std 0.5 px)
# H_true is only here to make the data; do not use it in your estimate.
rng = np.random.default_rng(5)
H_true = np.array([[1.2, 0.1, -50], [0.05, 0.9, 20], [2e-4, -1e-4, 1]])
src = rng.uniform(0, 1000, size=(30, 2))
h = np.column_stack([src, np.ones(30)]) @ H_true.T
dst = h[:, :2] / h[:, 2:3] + rng.normal(0, 0.5, size=(30, 2))

In [6]:
# Exercise 3: your solution


### Exercise 4

The provided `dst_bad` has 3 wrong correspondences. First run your plain DLT on all 30 points, then wrap it in RANSAC (4-point samples, threshold 3 px, refit on the inliers).

> **Expected result:** Plain DLT: mean error on the 27 good points $\approx10.4$ px (ruined). RANSAC: 27 inliers, rejects rows `[3, 11, 20]`, mean error on the inliers $\approx0.53$ px.

In [7]:
# --- provided data (run me first) ---
# dst_bad : copy of dst where rows 3, 11, 20 are moved by 50-100 px (wrong matches)
dst_bad = dst.copy()
dst_bad[[3, 11, 20]] += np.array([[80, -40], [-60, 90], [50, 70]])

In [8]:
# Exercise 4: your solution
